### Fucntion

In [1]:
def load_session_data(subject, date):
    """Load all data for a given subject and date"""
    import sys
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_utils.NeuralDataLoader import NeuralDataLoader, Dots3DMPConfig
    
    # Load session
    loader = NeuralDataLoader()
    loader.load_session(subject, date)
    config = Dots3DMPConfig(subject)

    # spike data (unit, trial, time)
    stimOn_spikes = loader.get_spike_data(alignment='stimOn', good_units_only=True, good_trials_only=True)
    saccOnset_spikes = loader.get_spike_data(alignment='saccOnset', good_units_only=True, good_trials_only=True)
    postTargHold_spikes = loader.get_spike_data(alignment='postTargHold', good_units_only=True, good_trials_only=True)
    tuning_spikes = loader.get_tuning_data(good_units_only=True, good_trials_only=True)

    # behavioral data
    behavior_dots3DMP = loader.get_behavioral_data(task='dots3DMP', good_trials_only=True, cal_mean_RT=True)
    behavior_tuning = loader.get_behavioral_data(task='tuning', good_trials_only=True)
    behavior_converted = config.convert_behavioral_data(behavior_dots3DMP, task='dots3DMP')
    behavior_tuning_converted = config.convert_behavioral_data(behavior_tuning, task='tuning')

    # Unit Info
    unit_info = loader.get_unit_info(good_units_only=True)
    MST_units = loader.get_units_by_area(unit_info, area_name='MST')
    VPS_units = loader.get_units_by_area(unit_info, area_name='VPS')
    MT_units = loader.get_units_by_area(unit_info, area_name='MT')
    dual_units = loader.get_units_by_area(unit_info, area_name='dual')

    # Time Info
    time_info = config.get_time_Info('dots3DMP')
    time_info_tuning = config.get_time_Info('tuning')
    time_axes_dots3DMP = config.get_time_axes('dots3DMP')
    time_axes_tuning = config.get_time_axes('tuning')

    # Prepare data
    spikes_data = {
        'stimOn': stimOn_spikes,
        'saccOnset': saccOnset_spikes,
        'postTargHold': postTargHold_spikes,
    }

    tuning_spikes_data = {'stimOn': tuning_spikes}
    
    units_data = {
        'MST': MST_units,
        'VPS': VPS_units,
        'MT': MT_units,
        'dual': dual_units
    }
    
    return {
        'loader': loader,
        'config': config,
        'spikes_data': spikes_data,
        'tuning_spikes_data': tuning_spikes_data,
        'behavior_converted': behavior_converted,
        'behavior_tuning_converted': behavior_tuning_converted,
        'unit_info': unit_info,
        'units_data': units_data,
        'time_axes_dots3DMP': time_axes_dots3DMP,
        'time_axes_tuning': time_axes_tuning,  
        'time_info': time_info,
        'time_info_tuning': time_info_tuning,
    }


In [2]:
def run_partial_correlation_analysis_all_sessions(session_list, save_results=True):
    """
    Run partial correlation analysis for all sessions (COMPARABLE VERSION).
    
    Automatically computes all 5 subsets with comparable estimates:
    - 'all': All small heading trials
    - 'correct': Correct trials only
    - 'error': Error trials only
    - 'high_pdw': High confidence trials
    - 'low_pdw': Low confidence trials
    
    Parameters:
    -----------
    session_list : list of tuples
        Each tuple: (subject, date)
    save_results : bool
        Whether to save results to disk
        
    Returns:
    --------
    all_session_results : dict
        Dictionary with results for each session
    """
    import sys
    import numpy as np
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_single_neurons.sliding_partial_corr import PartialCorrelationAnalyzer
    
    all_session_results = {}
    failed_sessions = []
    
    print("="*80)
    print("PARTIAL CORRELATION ANALYSIS - ALL SESSIONS (COMPARABLE VERSION)")
    print("="*80)
    print(f"Processing {len(session_list)} sessions...")
    print(f"Subsets: all, correct, error, high_pdw, low_pdw")
    print(f"Alignments: stimOn, saccOnset, postTargHold")
    print(f"Conditions: mod1_coh1, mod2_coh1, mod2_coh2, mod3_coh1, mod3_coh2")
    print(f"\nKEY: All subsets use SAME β coefficients (residualized on all small heading trials)")
    print("="*80)
    
    for i, (subject, date) in enumerate(session_list):
        try:
            print(f"\n{'='*60}")
            print(f"Session {i+1}/{len(session_list)}: {subject} {date}")
            print(f"{'='*60}")
            
            # Load session data
            print("\nLoading session data...")
            session_data = load_session_data(subject, date)
            
            # Get unit counts by area
            n_MST = len(session_data['units_data']['MST'])
            n_VPS = len(session_data['units_data']['VPS'])
            n_MT = len(session_data['units_data']['MT'])
            n_dual = len(session_data['units_data']['dual'])
            total_units = session_data['spikes_data']['stimOn'].shape[0]
            
            print(f"Units loaded: {total_units} total")
            print(f"  MST: {n_MST}, VPS: {n_VPS}, MT: {n_MT}, dual: {n_dual}")
            
            # Initialize analyzer with session data
            analyzer = PartialCorrelationAnalyzer(
                subject=subject,
                date=date,
                session_data=session_data
            )
            
            # Run analysis (automatically computes all 5 subsets)
            print(f"\nRunning partial correlation analysis (comparable version)...")
            results = analyzer.run_partial_correlation_analysis(
                spikes_data=session_data['spikes_data'],
                behavior_data=session_data['behavior_converted'],
                time_axes=session_data['time_axes_dots3DMP'],
                area='all',
                valid_units=None,
                conditions=None,
                save_results=save_results,
                verbose=True
            )
            
            # Store results
            session_key = f"{subject}_{date}"
            all_session_results[session_key] = {
                'results': results,
                'n_units': total_units,
                'n_MST': n_MST,
                'n_VPS': n_VPS,
                'n_MT': n_MT,
                'n_dual': n_dual
            }
            
            print(f"\n✓ Session {session_key} completed successfully")
            
            # Print quick summary for each subset
            subset_names = ['all', 'correct', 'error', 'high_pdw', 'low_pdw']
            
            for subset_name in subset_names:
                print(f"\n  Subset: {subset_name}")
                
                for alignment in ['stimOn', 'saccOnset', 'postTargHold']:
                    if alignment not in results:
                        continue
                    
                    print(f"    {alignment}:")
                    for cond_key in ['mod1_coh1', 'mod2_coh2', 'mod3_coh2']:
                        if cond_key not in results[alignment]['conditions']:
                            continue
                        
                        cond_data = results[alignment]['conditions'][cond_key]
                        
                        if subset_name not in cond_data:
                            continue
                        
                        subset_data = cond_data[subset_name]
                        
                        # Extract metrics
                        n_trials = subset_data['n_trials'][0] if len(subset_data['n_trials']) > 0 else 0
                        h_mean = np.nanmean(subset_data['heading_mean'])
                        c_mean = np.nanmean(subset_data['choice_mean'])
                        
                        print(f"      {cond_key}: n={n_trials}, heading={h_mean:.4f}, choice={c_mean:.4f}")
            
        except Exception as e:
            print(f"\n✗ Failed to process {subject} {date}: {e}")
            import traceback
            traceback.print_exc()
            failed_sessions.append((subject, date))
            continue
    
    # Final summary
    print("\n" + "="*80)
    print("ANALYSIS COMPLETE")
    print("="*80)
    print(f"Successfully processed: {len(all_session_results)}/{len(session_list)} sessions")
    print(f"Subsets computed: all, correct, error, high_pdw, low_pdw")
    
    if failed_sessions:
        print(f"\nFailed sessions ({len(failed_sessions)}):")
        for subj, dt in failed_sessions:
            print(f"  - {subj}_{dt}")
    
    # Total unit counts
    total_units_all = sum([v['n_units'] for v in all_session_results.values()])
    total_MST = sum([v['n_MST'] for v in all_session_results.values()])
    total_VPS = sum([v['n_VPS'] for v in all_session_results.values()])
    total_MT = sum([v['n_MT'] for v in all_session_results.values()])
    total_dual = sum([v['n_dual'] for v in all_session_results.values()])
    
    print(f"\nTotal units analyzed: {total_units_all}")
    print(f"  MST: {total_MST}")
    print(f"  VPS: {total_VPS}")
    print(f"  MT: {total_MT}")
    print(f"  dual: {total_dual}")
    
    print("\n✅ All sessions complete!")
    
    return all_session_results


def load_and_combine_partial_correlation_results(session_list):
    """
    Load saved partial correlation results (COMPARABLE VERSION) and combine them.
    
    Each session has ONE file with all 5 subsets.
    
    Parameters:
    -----------
    session_list : list of tuples
        Each tuple: (subject, date)
        
    Returns:
    --------
    combined_data : dict
        Combined data structure with all sessions
        Structure: combined_data[alignment] = list of session data
    """
    import sys
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_single_neurons.sliding_partial_corr import PartialCorrelationAnalyzer
    from pathlib import Path
    import pandas as pd
    
    save_dir = Path(r'D:\Neural-Pipeline\results\analysis_single_neurons\dot3DMP_partialcorr')
    
    # Create structure: alignment -> sessions
    combined_data = {
        'stimOn': [],
        'saccOnset': [],
        'postTargHold': [],
        'unit_metadata': []
    }
    
    print("Loading saved partial correlation results (COMPARABLE VERSION)...")
    print("File pattern: *_partialcorr_comparable.pkl\n")
    
    for subject, date in session_list:
        session_key = f"{subject}_{date}"
        
        # NEW filename pattern (no filter_type suffix)
        filepath = save_dir / f"{session_key}_all_partialcorr_comparable.pkl"
        
        if filepath.exists():
            print(f"  Loading {session_key}...")
            results = PartialCorrelationAnalyzer.load_results(filepath)
            
            # Add session info to unit metadata
            if 'unit_metadata' in results:
                unit_meta = results['unit_metadata'].copy()
                unit_meta['subject'] = subject
                unit_meta['date'] = date
                unit_meta['session'] = session_key
                combined_data['unit_metadata'].append(unit_meta)
            
            # Combine results for each alignment
            for alignment in ['stimOn', 'saccOnset', 'postTargHold']:
                if alignment in results:
                    combined_data[alignment].append({
                        'session': session_key,
                        'subject': subject,
                        'date': date,
                        'data': results[alignment]
                    })
        else:
            print(f"  ⚠ File not found: {filepath.name}")
    
    # Combine unit metadata into single DataFrame
    print("\nCombining metadata...")
    if combined_data['unit_metadata']:
        combined_data['unit_metadata'] = pd.concat(
            combined_data['unit_metadata'], 
            ignore_index=True
        )
        n_units = len(combined_data['unit_metadata'])
        print(f"  Combined {n_units} units from {len(session_list)} sessions")
        print(f"  Unit distribution:")
        print(combined_data['unit_metadata']['area'].value_counts().to_string(indent='    '))
    
    # Print summary of loaded data
    n_sessions = len(combined_data['stimOn'])
    print(f"\n✓ Loaded {n_sessions} sessions")
    print(f"  Each session contains 5 subsets: all, correct, error, high_pdw, low_pdw")
    
    return combined_data


def compare_subsets_across_sessions(combined_data, subset_pairs=[('correct', 'error'), ('high_pdw', 'low_pdw')]):
    """
    Compare correlations between different subsets across all sessions.
    
    Parameters:
    -----------
    combined_data : dict
        Output from load_and_combine_partial_correlation_results
    subset_pairs : list of tuples
        Pairs of subsets to compare
        
    Returns:
    --------
    comparison_results : dict
        Statistics comparing subsets
    """
    import numpy as np
    from scipy.stats import ttest_rel, wilcoxon
    
    print("\n" + "="*80)
    print("COMPARING SUBSETS ACROSS SESSIONS")
    print("="*80)
    
    comparison_results = {}
    
    for subset_a, subset_b in subset_pairs:
        print(f"\n{'='*60}")
        print(f"Comparing: {subset_a} vs {subset_b}")
        print(f"{'='*60}")
        
        comparison_results[f"{subset_a}_vs_{subset_b}"] = {}
        
        for alignment in ['stimOn', 'saccOnset', 'postTargHold']:
            print(f"\n{alignment}:")
            
            if alignment not in combined_data or len(combined_data[alignment]) == 0:
                continue
            
            for cond_key in ['mod1_coh1', 'mod2_coh2', 'mod3_coh2']:
                # Collect data across sessions
                heading_corrs_a = []
                heading_corrs_b = []
                choice_corrs_a = []
                choice_corrs_b = []
                
                for sess_data in combined_data[alignment]:
                    if cond_key not in sess_data['data']['conditions']:
                        continue
                    
                    cond_data = sess_data['data']['conditions'][cond_key]
                    
                    if subset_a not in cond_data or subset_b not in cond_data:
                        continue
                    
                    # Get mean correlations for this session
                    h_mean_a = np.nanmean(cond_data[subset_a]['heading_mean'])
                    h_mean_b = np.nanmean(cond_data[subset_b]['heading_mean'])
                    c_mean_a = np.nanmean(cond_data[subset_a]['choice_mean'])
                    c_mean_b = np.nanmean(cond_data[subset_b]['choice_mean'])
                    
                    if not np.isnan(h_mean_a) and not np.isnan(h_mean_b):
                        heading_corrs_a.append(h_mean_a)
                        heading_corrs_b.append(h_mean_b)
                    
                    if not np.isnan(c_mean_a) and not np.isnan(c_mean_b):
                        choice_corrs_a.append(c_mean_a)
                        choice_corrs_b.append(c_mean_b)
                
                if len(heading_corrs_a) < 3:
                    continue
                
                # Paired t-test
                t_h, p_h = ttest_rel(heading_corrs_a, heading_corrs_b)
                t_c, p_c = ttest_rel(choice_corrs_a, choice_corrs_b)
                
                print(f"\n  {cond_key} (n={len(heading_corrs_a)} sessions):")
                print(f"    Heading: {subset_a} M={np.mean(heading_corrs_a):.4f} vs {subset_b} M={np.mean(heading_corrs_b):.4f}")
                print(f"             t={t_h:.3f}, p={p_h:.4f} {'***' if p_h < 0.001 else '**' if p_h < 0.01 else '*' if p_h < 0.05 else 'n.s.'}")
                
                print(f"    Choice:  {subset_a} M={np.mean(choice_corrs_a):.4f} vs {subset_b} M={np.mean(choice_corrs_b):.4f}")
                print(f"             t={t_c:.3f}, p={p_c:.4f} {'***' if p_c < 0.001 else '**' if p_c < 0.01 else '*' if p_c < 0.05 else 'n.s.'}")
                
                # Store results
                key = f"{alignment}_{cond_key}"
                comparison_results[f"{subset_a}_vs_{subset_b}"][key] = {
                    'n_sessions': len(heading_corrs_a),
                    'heading': {
                        f'{subset_a}_mean': np.mean(heading_corrs_a),
                        f'{subset_b}_mean': np.mean(heading_corrs_b),
                        't': t_h,
                        'p': p_h
                    },
                    'choice': {
                        f'{subset_a}_mean': np.mean(choice_corrs_a),
                        f'{subset_b}_mean': np.mean(choice_corrs_b),
                        't': t_c,
                        'p': p_c
                    }
                }
    
    return comparison_results

### main

In [3]:
### main - COMPARABLE VERSION
import numpy as np
import sys
sys.path.append(r'D:\Neural-Pipeline\source')

# ============================================================================
# SESSION CONFIGURATION
# ============================================================================
subject = 'zarya'
dates = ['20250306', '20250411', '20250417', '20250501', '20250523', '20250602', '20250702', '20250710']

# Create simple session list
session_list = [(subject, date) for date in dates]

print("Session list created:")
for i, (subj, dt) in enumerate(session_list):
    print(f"  {i+1}. {subj}_{dt}")

print("\nComparable version will compute 5 subsets:")
print("  1. all: All small heading trials")
print("  2. correct: Correct trials only")
print("  3. error: Error trials only")
print("  4. high_pdw: High confidence (PDW=1)")
print("  5. low_pdw: Low confidence (PDW=0)")
print("\nKEY: All subsets use SAME β coefficients (fair comparison)")

# ============================================================================
# RUN PARTIAL CORRELATION ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("RUNNING PARTIAL CORRELATION ANALYSIS (COMPARABLE VERSION)")
print("="*80)

all_results = run_partial_correlation_analysis_all_sessions(
    session_list=session_list,
    save_results=True
)

# # ============================================================================
# # LOAD AND COMBINE RESULTS
# # ============================================================================
# print("\n" + "="*80)
# print("LOADING AND COMBINING RESULTS")
# print("="*80)

# combined_results = load_and_combine_partial_correlation_results(
#     session_list=session_list
# )

# # ============================================================================
# # COMPARE SUBSETS
# # ============================================================================
# print("\n" + "="*80)
# print("STATISTICAL COMPARISONS")
# print("="*80)

# comparison_stats = compare_subsets_across_sessions(
#     combined_results,
#     subset_pairs=[
#         ('correct', 'error'),
#         ('high_pdw', 'low_pdw')
#     ]
# )

# # ============================================================================
# # QUICK SUMMARY
# # ============================================================================
# print("\n" + "="*80)
# print("SUMMARY")
# print("="*80)

# print("\nData structure:")
# print("  results[alignment]['conditions'][condition][subset_name]")
# print("\nSubsets available:")
# print("  - all: All small heading trials")
# print("  - correct: Correct trials")
# print("  - error: Error trials")
# print("  - high_pdw: High confidence")
# print("  - low_pdw: Low confidence")

# print("\n✅ Partial correlation analysis (comparable version) complete!")

Session list created:
  1. zarya_20250306
  2. zarya_20250411
  3. zarya_20250417
  4. zarya_20250501
  5. zarya_20250523
  6. zarya_20250602
  7. zarya_20250702
  8. zarya_20250710

Comparable version will compute 5 subsets:
  1. all: All small heading trials
  2. correct: Correct trials only
  3. error: Error trials only
  4. high_pdw: High confidence (PDW=1)
  5. low_pdw: Low confidence (PDW=0)

KEY: All subsets use SAME β coefficients (fair comparison)

RUNNING PARTIAL CORRELATION ANALYSIS (COMPARABLE VERSION)
PARTIAL CORRELATION ANALYSIS - ALL SESSIONS (COMPARABLE VERSION)
Processing 8 sessions...
Subsets: all, correct, error, high_pdw, low_pdw
Alignments: stimOn, saccOnset, postTargHold
Conditions: mod1_coh1, mod2_coh1, mod2_coh2, mod3_coh1, mod3_coh2

KEY: All subsets use SAME β coefficients (residualized on all small heading trials)

Session 1/8: zarya 20250306

Loading session data...
Loaded dots3DMP data: zarya20250306dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20